In [1]:
print("test")
from platform import python_version

print(python_version())

test
3.11.11


In [2]:
import sys

# Confirm that we're using Python 3
assert sys.version_info.major == 3, 'Oops, not running Python 3. Use Runtime > Change runtime type'

# TensorFlow and tf.keras
import tensorflow as tf
from tensorflow import keras

# Helper libraries
import numpy as np
import matplotlib.pyplot as plt
import os
import subprocess

print('TensorFlow version: {}'.format(tf.__version__))

2025-07-16 12:30:49.209310: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-07-16 12:30:49.209349: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-07-16 12:30:49.210549: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-07-16 12:30:49.217159: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-07-16 12:30:49.827558: W tensorflow/comp

TensorFlow version: 2.15.0


In [3]:
# TensorFlow and tf.keras
import tensorflow as tf
from tensorflow import keras


fashion_mnist = keras.datasets.fashion_mnist
(train_images, train_labels), (test_images, test_labels) = fashion_mnist.load_data()

# scale the values to 0.0 to 1.0
train_images = train_images / 255.0
test_images = test_images / 255.0

# reshape for feeding into the model
train_images = train_images.reshape(train_images.shape[0], 28, 28, 1)
test_images = test_images.reshape(test_images.shape[0], 28, 28, 1)

class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

print('\ntrain_images.shape: {}, of {}'.format(train_images.shape, train_images.dtype))
print('test_images.shape: {}, of {}'.format(test_images.shape, test_images.dtype))


train_images.shape: (60000, 28, 28, 1), of float64
test_images.shape: (10000, 28, 28, 1), of float64


In [4]:
import numpy as np

# Numero totale di classi (Fashion MNIST ha 10 classi: 0–9)
NUM_CLASSES = 10
HALF = NUM_CLASSES // 2   # =5

# Primo sottoinsieme di classi: {0,1,2,3,4}
classes_1 = list(range(0, HALF))
# Secondo sottoinsieme di classi: {5,6,7,8,9}
classes_2 = list(range(HALF, NUM_CLASSES))

# --- Funzione per estrarre esempi la cui label è in `class_list`, rimappando le etichette su 0..(len(class_list)-1) ---
def extract_subset(images, labels, class_list):
    """
    Filtra images, labels tenendo solo le etichette presenti in class_list;
    rimappa le etichette in un range 0..(len(class_list)-1).
    """
    mask = np.isin(labels, class_list)
    imgs_sub = images[mask]
    lbls_sub = labels[mask]
    # Rimappo ogni label x in label_index[x] dove label_index mappa class_list su 0..len-1
    label_index = {c:i for i,c in enumerate(class_list)}
    lbls_sub_mapped = np.vectorize(lambda x: label_index[x])(lbls_sub)
    return imgs_sub, lbls_sub_mapped

# Estrazione per il “client 1” (classi 0–4)
train_images_1, train_labels_1 = extract_subset(train_images, train_labels, classes_1)
test_images_1,  test_labels_1  = extract_subset(test_images,  test_labels,  classes_1)

# Estrazione per il “client 2” (classi 5–9)
train_images_2, train_labels_2 = extract_subset(train_images, train_labels, classes_2)
test_images_2,  test_labels_2  = extract_subset(test_images,  test_labels,  classes_2)

print("Client 1 – esempi di train:", train_images_1.shape, train_labels_1.shape)
print("Client 2 – esempi di train:", train_images_2.shape, train_labels_2.shape)


Client 1 – esempi di train: (30000, 28, 28, 1) (30000,)
Client 2 – esempi di train: (30000, 28, 28, 1) (30000,)


In [5]:
import tensorflow as tf
from tensorflow import keras

def create_local_model(num_local_classes=5):
    """
    Restituisce un modello convolutional semplice con output = num_local_classes.
    """
    m = keras.Sequential([
        keras.layers.Conv2D(input_shape=(28,28,1), filters=8, kernel_size=3, 
                            strides=2, activation='relu', name='Conv1'),
        keras.layers.Flatten(),
        keras.layers.Dense(num_local_classes, activation='softmax', name='Softmax')
    ])
    return m

# Creo i due modelli
model_1 = create_local_model(num_local_classes=HALF)   # output=5, per classi [0-4]
model_2 = create_local_model(num_local_classes=HALF)   # output=5, per classi [5-9]

print("=== Model 1 (classi 0..4) ===")
model_1.summary()
print("\n=== Model 2 (classi 5..9) ===")
model_2.summary()


=== Model 1 (classi 0..4) ===
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 Conv1 (Conv2D)              (None, 13, 13, 8)         80        
                                                                 
 flatten (Flatten)           (None, 1352)              0         
                                                                 
 Softmax (Dense)             (None, 5)                 6765      
                                                                 
Total params: 6845 (26.74 KB)


2025-07-16 12:30:52.384425: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-07-16 12:30:52.428345: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-07-16 12:30:52.430780: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

Trainable params: 6845 (26.74 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________

=== Model 2 (classi 5..9) ===
Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 Conv1 (Conv2D)              (None, 13, 13, 8)         80        
                                                                 
 flatten_1 (Flatten)         (None, 1352)              0         
                                                                 
 Softmax (Dense)             (None, 5)                 6765      
                                                                 
Total params: 6845 (26.74 KB)
Trainable params: 6845 (26.74 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [6]:
# Numero di epoche
EPOCHS_LOCAL = 10   # puoi aumentare se vuoi

# ---> Model 1 su train_images_1 / train_labels_1 <---
model_1.compile(optimizer='adam',
                loss='sparse_categorical_crossentropy',
                metrics=['accuracy'])
model_1.fit(train_images_1, train_labels_1, epochs=EPOCHS_LOCAL, validation_split=0.1)
loss1, acc1 = model_1.evaluate(test_images_1, test_labels_1)
print(f"\nModel1 (classi 0–4) – Test accuracy: {acc1:.4f}")

# ---> Model 2 su train_images_2 / train_labels_2 <---
model_2.compile(optimizer='adam',
                loss='sparse_categorical_crossentropy',
                metrics=['accuracy'])
model_2.fit(train_images_2, train_labels_2, epochs=EPOCHS_LOCAL, validation_split=0.1)
loss2, acc2 = model_2.evaluate(test_images_2, test_labels_2)
print(f"\nModel2 (classi 5–9) – Test accuracy: {acc2:.4f}")


2025-07-16 12:30:53.083156: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 84672000 exceeds 10% of free system memory.
2025-07-16 12:30:53.153790: I external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:1101] failed to allocate 13.46MiB (14109696 bytes) from device: CUDA_ERROR_OUT_OF_MEMORY: out of memory
2025-07-16 12:30:53.153848: I external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:1101] failed to allocate 13.46MiB (14109696 bytes) from device: CUDA_ERROR_OUT_OF_MEMORY: out of memory
2025-07-16 12:31:03.154086: I external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:1101] failed to allocate 13.46MiB (14109696 bytes) from device: CUDA_ERROR_OUT_OF_MEMORY: out of memory
2025-07-16 12:31:03.154145: I external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:1101] failed to allocate 13.46MiB (14109696 bytes) from device: CUDA_ERROR_OUT_OF_MEMORY: out of memory
2025-07-16 12:31:03.154156: W external/local_tsl/tsl/framework/bfc_allocator.cc:4

InternalError: Failed copying input tensor from /job:localhost/replica:0/task:0/device:CPU:0 to /job:localhost/replica:0/task:0/device:GPU:0 in order to run _EagerConst: Dst tensor is not initialized.

In [ ]:
import numpy as np
import tensorflow as tf
import tensorflow_federated as tff
from tensorflow import keras
from tensorflow_federated.learning.models import from_keras_model
from tensorflow_federated.learning.algorithms import build_weighted_fed_avg
from tensorflow_federated.learning.optimizers import build_adam, build_sgdm

# 1) Definisci model_fn usando from_keras_model (non più tff.learning.from_keras_model)
def model_fn():
    keras_model = create_local_model(num_local_classes=NUM_CLASSES)
    return from_keras_model(
        keras_model=keras_model,
        input_spec=input_spec,
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=[tf.keras.metrics.SparseCategoricalAccuracy()],
    )  # :contentReference[oaicite:0]{index=0}

# 2) Costruisci il processo di FedAvg con la nuova API
iterative_process = build_weighted_fed_avg(
    model_fn=model_fn,
    client_optimizer_fn=lambda: build_adam(learning_rate=0.01),
    server_optimizer_fn=lambda: build_sgdm(learning_rate=1.0),
)  # :contentReference[oaicite:1]{index=1}

# 3) Inizializza e fai girare i round
state = iterative_process.initialize()
NUM_ROUNDS = 20

for rnd in range(1, NUM_ROUNDS + 1):
    state, metrics = iterative_process.next(state, [train_dataset_1, train_dataset_2])
    print(f"Round {rnd:02d} — loss={metrics.loss:.4f}, acc={metrics.sparse_categorical_accuracy:.4f}")

# 4) Estrai i pesi e assegnali al tuo Keras model
global_model = create_local_model(num_local_classes=NUM_CLASSES)
model_weights = iterative_process.get_model_weights(state)
model_weights.assign_weights_to(global_model)  # :contentReference[oaicite:2]{index=2}

# Ora `global_model` è il modello federato addestrato su tutte e 10 le classi.
